# E2 · The ColumnTransformer that silently drops the most important columns

**Type**: Find the bug · **Level**: ●●○ independent · **Time**: about 30 minutes · **core path ★**

## Objective

Find out why a model on Titanic has an AUC of 0.7 instead of 0.85 even though there are no errors: a `ColumnTransformer` with
`remainder="drop"` (the default) dropped `sex`, `title` and `pclass`. Rebuild the lost columns and fix it.

## Dataset

`ds.titanic()` with the usual 75/25 split. The colleague's preprocessor handles only `["age", "siblings_spouses", "parents_children",
"fare"]` as numeric and `["port", "deck"]` as categorical.

## Educational purpose

Recognize one of the most insidious scikit-learn bugs: **columns not listed disappear without warning**. You will learn to inspect
`transformers_` to see what was dropped, and to put a check in the pipeline.

## Task

1. Run the colleague's cell: `cv_auc_bug` is far below what you would expect from Titanic.
2. Inspect `ct.transformers_` of the trained preprocessor: the last entry (`"remainder"`) contains the **indices** of the dropped columns. Save in `lost_columns` the sorted list of their names (`X.columns[indices]`).
3. Fix the preprocessor: add the three missing columns to the right group (`sex`, `title` to the categoricals; `pclass` in `passthrough`) and save `cv_auc_ok` (same `cv` and same model).
4. Write `check_columns(ct, X)` that raises `ValueError` if any column of `X` ended up in `remainder` without it having been decided (that is if `ct.transformers_[-1][0] == "remainder"` and it has columns). Check that it raises for the colleague's preprocessor and not for the correct one.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = ds.titanic()
X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
NUM = ["age", "siblings_spouses", "parents_children", "fare"]
CAT = ["sex", "port", "deck", "title"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)

ct_bug = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["age", "siblings_spouses", "parents_children", "fare"]),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), ["port", "deck"]),
], sparse_threshold=0)

## Your work

### The colleague's code

In [ ]:
bug_model = make_pipeline(ct_bug, LogisticRegression(max_iter=2000))
cv_auc_bug = cross_val_score(bug_model, X_tr, y_tr, cv=cv, scoring="roc_auc").mean()
print(f"cross-validated AUC: {cv_auc_bug:.3f}")

### Your diagnosis

In [ ]:
ct_bug.fit(X_tr, y_tr)
lost_columns = None

In [ ]:
ct_ok = None
cv_auc_ok = cv_auc_bug

In [ ]:
def check_columns(ct, X):
    ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`name, transformer, indices = ct_bug.transformers_[-1]` (name `"remainder"`); `list(X_tr.columns[indices])`. For the check: `if ct.transformers_[-1][0] == "remainder" and len(ct.transformers_[-1][2]) > 0: raise ValueError(...)`.'):
    check(cv_auc_bug < 0.78, "the colleague's model is far below expectations (AUC below 0.78)", "don't modify the colleague's code")
    check(lost_columns == ["pclass", "sex", "title"], "dropped columns: pclass, sex, title", "indices in ct_bug.transformers_[-1][2], names with X.columns[indices]")
    check(cv_auc_ok > cv_auc_bug + 0.08 and cv_auc_ok > 0.82, "including the lost columns the AUC goes up by more than 0.08 (above 0.82)",
          "add sex and title to the categoricals and pclass in passthrough")
    try:
        check_columns(ct_bug, X_tr)
        raises_bug = False
    except ValueError:
        raises_bug = True
    ct_ok.fit(X_tr, y_tr)
    try:
        check_columns(ct_ok, X_tr)
        raises_ok = False
    except ValueError:
        raises_ok = True
    check(raises_bug and not raises_ok, "check_columns raises for the colleague's preprocessor and not for the correct one",
          "raise ValueError if ct.transformers_[-1][0] == 'remainder' and the list of indices is not empty")
done()

---
The commented solution is in `solutions/E_find_the_bug/E2_columntransformer_drops_columns.ipynb`. Open it after you pass the check, or after you have really tried.